# Test Functionality of GestionOt{class}

### Pasos para calificar una actividad.


1. Separar los eventos que tienen alimentador de los que no.
   
   1.1. Separar y calificar aquellos que son de TRANSPORTE, ALIMENTACIÓN, SE LABORA, INFO, se repite en la calificación
   
2. A los eventos que si tienen alimentador.
   
   2.1. Separar aquellos que sabemos que son SAPG, los más fáciles de identificar.

   2.2. Separar aquellos que son de Servicios Ocasionales.

   2.3. Calificar usando la Red Neuronal.

In [11]:

from eerssa import gestionOT
from eerssa import matrizActividades
from pprint import pprint
from pathlib import Path
import pandas as pd

#test_path = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/'
test_path = '/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/2025'

list_pdfs = []
for path in Path( test_path ).glob("**/*.pdf"):
  list_pdfs.append( str(path) )
  list_pdfs.sort()


### DASK

Primero creo un cluster locar de computación

In [2]:
from dask.distributed import LocalCluster
client = LocalCluster().get_client()

Luego, con el listado de OT's de ejecutado en primera instancia, genero objetos en los cluster de computación local, 

Ejecuto la función `load_ot()` y guardo los resultados a la misma lista de objetos

In [12]:

futures = [client.submit(gestionOT.GestionOt, file, actor=True) for file in list_pdfs ]
ot_array = [future.result() for future in futures]

ot_cargada = [ot.load_ot() for ot in ot_array]
obj_lists = [future.result() for future in ot_cargada]



🔥
> TODO
> Imprimir un reporte de las OT que no fue exitoso su conversion a OT. informar las novedades encontradas

🔥
> TODO 
> La siguiente linea de código es posible que no se este ejecutando en paralelo, verificarlo luego

Estoy simultaneamente, generando `matriz` que es un `Pandas.Dataframe` almacenandola en el objeto y en `ot_matrices`

In [13]:
ot_matrices = [ matrizActividades.ConvertirOT_a_ActividadesCSV(ot) for ot in obj_lists ]
df_total = [ df for df in ot_matrices if df is not None ]

/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarni

KeyError: 'actividades'

# TESTING

In [8]:
[print( f" Ot Link: {ot.link} | {ot.valido} ") for ot in obj_lists]

 Ot Link: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/10_Test Orden de trabajo Zamora 11-02-2022 (RM - Electricistas).pdf | True 
 Ot Link: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/11_LM_Tres_hojas.pdf | True 
 Ot Link: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/11_accidente canoa SI.pdf | True 
 Ot Link: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/12_NL_Tres_hojas_ultima_vacia.pdf | True 
 Ot Link: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/12_casoEspecial01.pdf | True 
 Ot Link: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/13_Orden de trabajo Guayzimi 18-07-2022 (CQ).pdf | True 
 Ot Link: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/14_Orden de trabajo Guayzimi 10-04-2022 (CQ).pdf | True 
 Ot Link: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/15_Orden de trabajo Zamora 20-04-2022 (FR).pdf | True 
 Ot Link: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/16_Orden

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [10]:
[type(df) for df in ot_matrices]

[pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame,
 pandas.core.frame.DataFrame]

### Generar la Matriz de Actividades para un objeto

In [7]:
nro_ot = 9
obj_lists[nro_ot].data['exito']

False

In [ ]:
obj_lists[nro_ot].data['actividades']

In [17]:
matriz_test = matrizActividades.ConvertirOT_a_ActividadesCSV(  obj_lists[nro_ot] )
matriz_test[['Cuenta','Evento','Tipo','Actividad','Alimentador','Fecha']]

,Cuenta,Evento,Tipo,Actividad,Alimentador,Fecha
0,informativa,"Loja , Subestación Obrapia asistencia capacita...",PREVENTIVO,PROG,None,2023-02-16 00:00:00
1,informativa,"LUNCH EN LOJA,",RUTINARIA,None,None,2023-02-16 00:00:00
2,informativa,Se continua en la Subestación Obrapia asistenc...,RUTINARIA,PROG,None,2023-02-16 00:00:00
3,se_labora,SE LABORA: PCH: de 08:00 13:00 y de 14:00 a 18...,None,LABORA,None,2023-02-16 00:00:00


In [15]:
dbg

version                                                     0.12.0
link             /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/te...
id_ot                                                     134508.0
exito                                                         True
cuadrilla                          Yacuambi Z1 (Cuadrilla. Nro. 8)
responsable                     [LOZANO SIGCHO NAUN ENRIQUE, JECE]
colaboradores    {'total': 3, 'nombres': [['CABRERA GONZALEZ LU...
diaSemana                                                    lunes
fecha                                    2024-07-29 00:00:00-05:00
fechaInicio                            lunes, 29 de julio del 2024
fechaFinal                                     29/07/2024 20:40:00
sitio                 Yacuambi - Tamboloma, Hucapamba y Jembuentza
descripcion      Traslado a Tamboloma para revisar sector sin s...
tEstimado                                                        8
vehiculo         {'numero': 'R-171', 'placa': 'AAA-4278', 'mar

### Secuencial GLOBAL LOCK

In [ ]:
### Secuencial en un solo procesador.
obj_lists = []
for file in list_pdfs:
  ot = gestionOT.GestionOt( file )
  ot.load_ot()
  obj_lists.append( ot )